# Phần 1. NumPy trong workflow ML/DL

Các bài dưới đây dùng dữ liệu nhỏ để mô phỏng preprocessing, inference và xử lý
tensor trong một pipeline thực tế.

In [ ]:
STUDENT_NAME = "Truong Vy Kiet"  # TODO: Họ và tên
STUDENT_ID = "2520005"    # TODO: MSSV

print(f"Student: {STUDENT_NAME} ({STUDENT_ID})")

Student: Truong Vy Kiet (2520005)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd '/content/drive/MyDrive/mliot-pyml-2026-hw/week02/numpy-pandas-eda-hw/data'

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.8)

DATA_CANDIDATES = [
    Path("week02/numpy-pandas-eda-hw/data/automobile_raw.csv"),
    Path("data/automobile_raw.csv"),
    Path("../data/automobile_raw.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Không tìm thấy data/automobile_raw.csv")

print("Data path:", DATA_PATH.resolve())

## N1. Stable softmax cho batch logits

Một classifier trả về `logits` có shape `(batch_size, num_classes)`. Tính softmax
theo từng mẫu bằng cách trừ giá trị lớn nhất trên mỗi hàng trước khi gọi `np.exp`.
Cách viết này tránh overflow khi logits có giá trị lớn.

**Biến đầu ra bắt buộc**

- `shifted_logits`: logits sau khi trừ row-wise maximum.
- `class_probabilities`: xác suất mỗi class, mỗi hàng có tổng bằng 1.
- `predicted_classes`: class có xác suất lớn nhất của từng mẫu.
- `confidence_scores`: xác suất lớn nhất của từng mẫu.

In [ ]:
logits = np.array([
    [2.0, 1.0, 0.1],
    [1000.0, 1001.0, 999.0],
    [-2.0, -1.0, 3.0],
    [0.5, 0.5, 0.5],
], dtype=np.float64)

In [ ]:
# TODO N1
row_max = np.max(logits, axis = 1, keepdims = True)
# shifted_logits = ...
shifted_logits = logits - row_max
# exp_logits = ...
exp_logits = np.exp(shifted_logits)
# exp_logits_sum = ...
exp_logits_sum = np.sum(exp_logits, axis = 1, keepdims = True)
# class_probabilities = ...
class_probabilities = exp_logits / exp_logits_sum
# predicted_classes = ...
predicted_classes = np.argmax(class_probabilities, axis = 1)
# confidence_scores = ...
confidence_scores = np.max(class_probabilities, axis = 1)

In [ ]:
required = [
    "shifted_logits",
    "class_probabilities",
    "predicted_classes",
    "confidence_scores",
]
if not all(name in globals() for name in required):
    print("Complete N1 to run this self-check.")
else:
    assert class_probabilities.shape == logits.shape
    assert np.all(np.isfinite(class_probabilities))
    assert np.allclose(class_probabilities.sum(axis=1), 1.0)
    assert predicted_classes.shape == (logits.shape[0],)
    assert confidence_scores.shape == (logits.shape[0],)
    print("N1 self-check passed")

## N2. Chuẩn hóa train và validation không gây leakage

Mỗi hàng là một mẫu, mỗi cột là một feature. Tính mean/std **chỉ từ `X_train`**,
sau đó dùng cùng thống kê để transform cả train và validation.

**Biến đầu ra bắt buộc**

- `train_feature_mean`, `train_feature_std`: shape `(4,)`.
- `X_train_scaled`: train set đã chuẩn hóa.
- `X_val_scaled`: validation set dùng thống kê từ train.

In [ ]:
# Features: height_cm, weight_kg, activity_hours, age
X_train = np.array([
    [170.0, 65.0, 1.2, 22.0],
    [180.0, 80.0, 2.4, 35.0],
    [160.0, 50.0, 0.8, 19.0],
    [175.0, 70.0, 1.5, 28.0],
    [168.0, 60.0, 1.0, 24.0],
    [182.0, 90.0, 3.0, 41.0],
])

X_val = np.array([
    [172.0, 68.0, 1.4, 26.0],
    [190.0, 95.0, 3.4, 45.0],
])

In [ ]:
# TODO N2
# train_feature_mean = ...
train_feature_mean = np.mean(X_train, axis = 0)
# train_feature_std = ...
train_feature_std = np.std(X_train, axis = 0)
# X_train_scaled = ...
X_train_scaled = (X_train - train_feature_mean) / train_feature_std
# X_val_scaled = ...
X_val_scaled = (X_val - train_feature_mean) / train_feature_std

In [ ]:
required = [
    "train_feature_mean",
    "train_feature_std",
    "X_train_scaled",
    "X_val_scaled",
]
if not all(name in globals() for name in required):
    print("Complete N2 to run this self-check.")
else:
    assert X_train_scaled.shape == X_train.shape
    assert X_val_scaled.shape == X_val.shape
    assert np.allclose(X_train_scaled.mean(axis=0), 0.0)
    assert np.allclose(X_train_scaled.std(axis=0), 1.0)
    print("N2 self-check passed")

## N3. Tạo review queue sau inference

Giả sử `class_probabilities` đến từ N1. Một prediction cần được kiểm tra thủ công
nếu dự đoán sai **hoặc** confidence nhỏ hơn `0.70`.

**Biến đầu ra bắt buộc**

- `correct_mask`
- `high_confidence_mask`
- `review_mask`
- `review_indices`

In [ ]:
true_labels = np.array([0, 2, 2, 1])
confidence_threshold = 0.70

In [ ]:
from IPython.utils.py3compat import re
# TODO N3
# correct_mask = ...
correct_mask = (true_labels == predicted_classes)
# high_confidence_mask = ...
high_confidence_mask = (confidence_scores >= confidence_threshold)
# review_mask = ...
review_mask = ~(correct_mask | high_confidence_mask)
# review_indices = ...
review_indices = np.where(review_mask)[0]

## N4. Tiền xử lý và augment một batch ảnh

`image_batch_uint8` có layout `(B, H, W, C)`. Chuyển batch về `float32` trong đoạn
`[0, 1]`, sau đó tạo một batch mới được flip ngang. Batch augment phải có bộ nhớ
độc lập để việc chỉnh sửa không làm thay đổi batch đã normalize.

Sau khi tạo batch augment, đặt pixel `augmented_batch[0, 0, 0, 0] = 1.0`.

**Biến đầu ra bắt buộc:** `normalized_batch`, `augmented_batch`.

In [ ]:
image_batch_uint8 = (
    np.arange(2 * 4 * 4 * 3, dtype=np.uint8)
    .reshape(2, 4, 4, 3)
)

In [ ]:
# TODO N4
# normalized_batch = ...
normalized_batch = image_batch_uint8.astype(np.float32) / 255.0
# augmented_batch = ...
augmented_batch = normalized_batch[:, :, ::-1, :].copy()
# augmented_batch[0, 0, 0, 0] = ...
augmented_batch[0, 0, 0, 0] = 1.0

# Phần 2. EDA với Automobile

Đọc `data/data_dictionary.md` trước khi xử lý.

## Câu hỏi mở đầu

1. Mỗi dòng đại diện cho đối tượng gì?
2. Ký hiệu missing value trong CSV là gì?
3. `symboling` có ý nghĩa gì?

**Trả lời**

<!-- Viết câu trả lời tại đây. -->
1. Mỗi dòng đại diện cho thông số kỹ thuật, mức độ rủi ro bảo hiểm, tổn thất thực tế và giá thành của một mẫu xe ô tô cụ thể tại thị trường Mỹ.
2. Dấu chấm hỏi (?)
3. symboling là hệ sinh số chỉ mức độ rủi ro của xe gắn liền với bảo hiểm Auto. Giá trị dao động từ -3 đến +3. Giá trị càng lớn thể hiện xe càng rủi ro (nguy hiểm/dễ tai nạn hơn), giá trị âm thể hiện xe an toàn hơn về mặt bảo hiểm.

## D1. Load và inspect raw CSV

Load dữ liệu sao cho dấu `?` vẫn là chuỗi để quan sát ảnh hưởng tới dtype.

**Biến đầu ra bắt buộc**

- `raw_df`: DataFrame raw.
- `raw_shape`: tuple.
- `raw_missing_marker_count`: tổng số dấu `?`.

In [ ]:
# TODO D1
# raw_df = ...
raw_df = pd.read_csv('automobile_raw.csv')
# raw_shape = ...
raw_shape = raw_df.shape
# raw_missing_marker_count = ...
raw_missing_marker_count = (raw_df == '?').sum().sum()

## D2. Missing values và dtype

1. Thay `?` bằng `np.nan`.
2. Chuyển các cột trong `NUMERIC_COLUMNS` bằng `pd.to_numeric`.
3. Tạo báo cáo missing.

**Biến đầu ra bắt buộc:** `df_clean`, `missing_by_column`.

In [ ]:
NUMERIC_COLUMNS = ['symboling', 'normalized_losses', 'wheel_base', 'length', 'width', 'height', 'curb_weight', 'engine_size', 'bore', 'stroke', 'compression_ratio', 'horsepower', 'peak_rpm', 'city_mpg', 'highway_mpg', 'price']

In [ ]:
# TODO D2
# df_clean = ...
df_clean = raw_df.replace('?', np.nan)
# for column in NUMERIC_COLUMNS:
#     ...
# missing_by_column = ...
for column in NUMERIC_COLUMNS:
    df_clean[column] = pd.to_numeric(df_clean[column], errors='coerce')
missing_by_column = df_clean.isnull().sum()

### Giải thích cách làm sạch dữ liệu

- Vì sao không nên fill tất cả numeric columns bằng cùng một giá trị?
- Với `price`, lựa chọn drop hay fill phù hợp hơn cho bài EDA này? Vì sao?
- `normalized_losses` thiếu nhiều dữ liệu hơn các cột khác. Điều này ảnh hưởng thế nào?

**Nhận xét**

<!-- Viết 3--6 câu tại đây. -->

## D3. DataFrame sang NumPy

Dùng sáu cột trong `AUTO_FEATURES`. Drop các dòng thiếu ít nhất một trong
sáu cột, sau đó chuyển sang `float64` NumPy array và chuẩn hóa theo feature.

**Biến đầu ra bắt buộc**

- `analysis_df`
- `X_auto`
- `auto_feature_mean`
- `auto_feature_std`
- `X_auto_scaled`

In [ ]:
AUTO_FEATURES = ['curb_weight', 'engine_size', 'horsepower', 'city_mpg', 'highway_mpg', 'price']

In [ ]:
# TODO D3
# analysis_df = ...
analysis_df = df_clean[AUTO_FEATURES].dropna()
# X_auto = ...
X_auto = analysis_df.values.astype(np.float64)
# auto_feature_mean = ...
auto_feature_mean = np.mean(X_auto, axis=0)
# auto_feature_std = ...
auto_feature_std = np.std(X_auto, axis=0)
# X_auto_scaled = ...
X_auto_scaled = (X_auto - auto_feature_mean) / auto_feature_std

## D4. Outlier theo price z-score

Tính z-score của `price` bằng NumPy. Một dòng được xem là outlier trong bài
này khi `abs(z) > 2`.

**Biến đầu ra bắt buộc:** `price_z`, `price_outlier_mask`, `price_outliers`.

In [ ]:
# TODO D4
# price_index = ...
price_index = AUTO_FEATURES.index('price')
# price_z = ...
price_values = X_auto[:, price_index]
price_z = (price_values - auto_feature_mean[price_index]) / auto_feature_std[price_index]
# price_outlier_mask = ...
price_outlier_mask = np.abs(price_z) > 2
# price_outliers = ...
price_outliers = X_auto[price_outlier_mask]

## D5. Correlation và GroupBy

**Biến đầu ra bắt buộc**

- `engine_price_corr`: Pearson correlation tính bằng NumPy.
- `price_by_body_style`: Series mean price theo `body_style`, sort index.

In [ ]:
# TODO D5
engine_size_vals = X_auto[:, AUTO_FEATURES.index('engine_size')]
price_vals = X_auto[:, AUTO_FEATURES.index('price')]
# engine_price_corr = ...
engine_price_corr = np.corrcoef(engine_size_vals, price_vals)[0, 1]
# price_by_body_style = ...
price_by_body_style = df_clean.groupby('body_style')['price'].mean().sort_index()

# Phần 3. Visualization và insight

Mỗi biểu đồ cần:

1. một câu hỏi;
2. title, axis labels và unit;
3. lựa chọn chart phù hợp;
4. 1--2 câu nhận xét ngay dưới chart.

## M2.1 Price phân phối như thế nào?

In [ ]:
# TODO M2.1: histogram/KDE của price
sns.histplot(data=df_clean, x='price', kde=True, color='teal')
plt.title('Phân phối giá xe ô tô (Price Distribution)')
plt.xlabel('Giá xe (USD)')
plt.ylabel('Số lượng xe')
plt.show()

**Nhận xét:** <!-- 1--2 câu -->

Biểu đồ phân phối giá xe bị lệch phải (skewed right) rất mạnh. Hầu hết các mẫu xe tập trung ở phân khúc giá rẻ và bình dân từ 5,000 USD đến 15,000 USD, trong khi số lượng xe cao cấp giá trên 30,000 USD rất ít.

## M2.2 Dataset có cân bằng theo body style không?

In [ ]:
# TODO M2.2: countplot của body_style
sns.countplot(data=df_clean, x='body_style', order=df_clean['body_style'].value_counts().index, palette='viridis')
plt.title('Thống kê số lượng xe theo kiểu dáng thân xe (Body Style)')
plt.xlabel('Kiểu dáng xe')
plt.ylabel('Số lượng mẫu xe')
plt.show()

**Nhận xét:** <!-- 1--2 câu -->

## M2.3 Price khác nhau theo body style ra sao?

In [ ]:
# TODO M2.3: boxplot price theo body_style
sns.boxplot(data=df_clean, x='body_style', y='price', palette='Set2')
plt.title('Phân vị giá xe theo từng kiểu dáng thân xe')
plt.xlabel('Kiểu dáng xe')
plt.ylabel('Giá bán (USD)')
plt.show()

**Nhận xét:** <!-- 1--2 câu -->

Dòng xe hardtop và convertible có dải giá bán trung vị cao hơn đáng kể so với các dòng xe thông dụng. Xe dạng hatchback có dải giá hẹp nhất và mức giá trung bình rẻ nhất trong các loại.

## M2.4 Engine size liên quan thế nào tới price?

In [ ]:
# TODO M2.4: scatterplot engine_size và price, hue=fuel_type
sns.scatterplot(data=df_clean, x='engine_size', y='price', hue='fuel_type', alpha=0.8)
plt.title('Mối quan hệ giữa kích thước động cơ và giá xe')
plt.xlabel('Kích thước động cơ (Engine Size - CID)')
plt.ylabel('Giá xe (USD)')
plt.show()

**Nhận xét:** <!-- 1--2 câu -->

Giữa engine_size và price có một mối tương quan tuyến tính thuận rất rõ ràng: động cơ càng lớn thì giá xe càng đắt. Các xe sử dụng nhiên liệu Diesel hay Gas đều tuân theo xu hướng chung này.

## M2.5 Các feature numeric tương quan ra sao?

In [ ]:
# TODO M2.5: correlation heatmap
corr_matrix = df_clean[AUTO_FEATURES].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Ma trận hệ số tương quan giữa các đặc trưng số')
plt.show()

**Nhận xét:** <!-- 1--2 câu -->

Trọng lượng xe (curb_weight), engine_size và horsepower có tương quan thuận cực kỳ mạnh với giá bán (price) (đều > 0.80). Ngược lại, mức tiêu hao nhiên liệu (city_mpg, highway_mpg) có tỷ lệ nghịch (tương quan âm) rất lớn với giá thành, xe càng đắt/nặng thì càng tốn nhiên liệu.

## M2.6 Biểu đồ tự chọn

Đặt một câu hỏi mới, chọn chart phù hợp và không lặp nguyên năm biểu đồ trên.

In [ ]:
# TODO M2.6: biểu đồ tự chọn
sns.boxplot(data=df_clean, x='aspiration', y='horsepower', palette='Pastel1')
plt.title('Sự khác biệt về mã lực (Horsepower) dựa trên hệ thống nạp khí')
plt.xlabel('Hệ thống nạp khí (Aspiration)')
plt.ylabel('Mã lực (Horsepower)')
plt.show()

**Nhận xét:** <!-- 1--2 câu -->

Động cơ trang bị bộ tăng áp (turbo) nhìn chung có dải mã lực trung vị dịch chuyển hẳn lên mức cao hơn và tập trung nhiều ở dải từ 100 đến 160 mã lực so với các xe nạp khí tự nhiên (std).

# Tổng hợp

Viết:

- 3--5 phát hiện chính có dẫn chứng;
- ít nhất 2 hạn chế của dataset;
- một ví dụ về correlation không đồng nghĩa causation;
- một câu hỏi nên phân tích tiếp.

## Tổng hợp của sinh viên

<!-- Viết khoảng 150--250 từ. -->

Phát hiện chính (có dẫn chứng): 1. Kích thước động cơ và Trọng lượng xe là hai yếu tố có tỷ lệ thuận lớn nhất đến giá xe với hệ số tương quan tuyến tính ghi nhận tương ứng đạt khoảng 0.87 và 0.83.
2. Mức tiêu thụ nhiên liệu trong thành phố tỷ lệ nghịch mạnh với giá xe (-0.69), chứng tỏ các dòng xe hạng sang đắt tiền thường tiêu thụ nhiên liệu tốn hơn.
3. Giá xe bị chi phối lớn bởi kiểu dáng, trong đó phân khúc mui trần luôn nằm ở nhóm xa xỉ, có giá thành trung vị vượt trội so với hatchback hay sedan phổ thông.

Hạn chế của dataset: 1. Tập dữ liệu chứa lượng dữ liệu thiếu khá lớn ở cột normalized_losses gây khó khăn cho việc phân tích toàn diện rủi ro bảo hiểm xe.
2. Lượng mẫu phân bổ không đồng đều (mất cân bằng nhóm nặng), có quá nhiều dòng xe Sedan/Hatchback nhưng lại quá ít dòng xe Hardtop hay Convertible, dễ gây thiên kiến khi huấn luyện mô hình.

Correlation không đồng nghĩa Causation: Trọng lượng xe lớn tương quan rất mạnh với giá xe cao. Tuy nhiên, việc "làm cho một chiếc xe nặng thêm bằng cách bỏ thêm sắt đá vào" không hề giúp tăng giá trị bán lẻ của xe. Thực tế, xe đắt hơn vì nó được trang bị động cơ lớn hơn, nội thất tiện nghi phức tạp hơn, kéo theo trọng lượng tăng lên một cách thụ động.

Câu hỏi nghiên cứu tiếp theo: Liệu các xe có chỉ số rủi ro bảo hiểm cao (symboling từ +2 đến +3) có phải là những dòng xe thể thao có công suất mã lực cực đại lớn và có giá thành đắt đỏ hay không?